# NERSC through the AmSC RIG

This notebook uses the **experimental** RIG helper in `amsc-client==0.7.1`.

```text
AMSC_TOKEN (AmSC Keycard/PAT) -> RIG -> facility IRI API
```

This is distinct from direct `client.facility(...)`, which uses facility-native authentication. RIG may still require an active facility allocation and configured Credential Vault or token-exchange access. Never paste, print, decode, compare, serialize, or commit a token.

**Validation status (2026-09-28):** this notebook is **statically validated** against the published wheel. The protected reads and optional workflow are **not live-validated** by this repository campaign. Submission is disabled by default.

This example discovers NERSC, proves the routed credential with a protected project read, inspects compute/homes resources, and provides an optional bounded Perlmutter workflow.

In [ ]:
import os
from importlib.metadata import version
from amsc_client import Client
from amsc_client.core.exceptions import ApiError, AuthenticationError

STAGING_API = "https://api.staging.american-science-cloud.org/api/current"
STAGING_RIG = "https://rig.staging.american-science-cloud.org"
AMSC_TOKEN = os.environ["AMSC_TOKEN"]
assert version("amsc-client") == "0.7.1", "Install requirements.txt and restart this kernel"
client = Client(token=AMSC_TOKEN, base_url=STAGING_API)
print("Client release:", version("amsc-client"))

## Discover and connect
NERSC through RIG may use Tier-3 vaulted facility credentials. Do not paste a NERSC bearer into this notebook.

In [ ]:
facilities = client.rig_facilities(rig_url=STAGING_RIG)
nersc_meta = next((f for f in facilities if f.name.lower() == "nersc"), None)
if nersc_meta is None:
    raise RuntimeError("NERSC is not advertised by this RIG deployment")
print(nersc_meta.name, nersc_meta.display_name, nersc_meta.tier, nersc_meta.api_version)
nersc = client.facility_via_rig("nersc", rig_url=STAGING_RIG)

## Protected read, then resource discovery
A successful `projects()` response proves routed authentication; public discovery alone does not.

In [ ]:
projects = nersc.projects()
print(f"Protected project read succeeded ({len(projects)} visible project(s))")
resources = nersc.resources()
for resource in resources:
    print(resource.name, resource.resource_type, resource.status)
compute = nersc.resource("compute")
homes = nersc.resource("homes")

## Optional Perlmutter submission (default off)
Set NERSC environment values before enabling the gate. The output path is the existing NERSC home root and uses a run-unique filename.

In [ ]:
from datetime import datetime, timezone

SUBMIT_JOB = False
NERSC_USERNAME = os.getenv("NERSC_USERNAME", "")
NERSC_ACCOUNT = os.getenv("NERSC_ACCOUNT", "")
NERSC_QUEUE = os.getenv("NERSC_QUEUE", "debug")
NERSC_CONSTRAINT = os.getenv("NERSC_CONSTRAINT", "cpu")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
HOME_LETTER = NERSC_USERNAME[:1]
OUTPUT_DIR = f"/global/homes/{HOME_LETTER}/{NERSC_USERNAME}"
JOB_NAME = f"amsc-rig-nersc-{RUN_ID}"

if SUBMIT_JOB:
    required = {"NERSC_USERNAME": NERSC_USERNAME, "NERSC_ACCOUNT": NERSC_ACCOUNT, "NERSC_QUEUE": NERSC_QUEUE, "NERSC_CONSTRAINT": NERSC_CONSTRAINT}
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise RuntimeError("Missing required environment settings: " + ", ".join(missing))
    stdout_path = f"{OUTPUT_DIR}/{JOB_NAME}.stdout"
    stderr_path = f"{OUTPUT_DIR}/{JOB_NAME}.stderr"
    job = compute.submit(
        executable="/bin/echo", arguments=["Hello from AmSC through RIG", RUN_ID],
        directory=OUTPUT_DIR, name=JOB_NAME, queue=NERSC_QUEUE, account=NERSC_ACCOUNT,
        duration=300, nodes=1, stdout_path=stdout_path, stderr_path=stderr_path,
        constraint=NERSC_CONSTRAINT,
    )
else:
    print("SUBMIT_JOB=False: no scheduler job was submitted.")

## Bounded wait and output read
This executes only if the default-off submission gate was enabled and submission succeeded.

In [ ]:
if SUBMIT_JOB:
    completed = job.wait(timeout=600, poll_interval=10)
    output_task = homes.fs.head(stdout_path, lines=50)
    print(output_task.result)